# Residency Day 2: Project Deliverable 2

## Regression Modeling and Performance Evaluation

This notebook builds and compares regression models to predict healthcare spending using engineered features from the CMS claims dataset. The workflow includes data cleaning, feature engineering, model training, cross-validation, and performance visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

DATA_PATH = Path('cms_healthcare_claims_cleaned.csv')
df = pd.read_csv(DATA_PATH)
df.head()


## Data Preparation and Feature Engineering

The original dataset is already cleaned, but several useful features can be engineered to improve regression performance. These include a year indicator, quarter indicator, claims-per-beneficiary ratio, and log-transformed versions of the main count variables.

In [ ]:
# Clean a few key columns and engineer decision-friendly features.
df = df.copy()

df['Gnrc_Name'] = df['Gnrc_Name'].fillna('Unknown')

# Extract year and quarter from the Year field such as 2025 (Q1-Q4)
df['Year_Value'] = pd.to_numeric(df['Year'].str.extract(r'(\d{4})')[0], errors='coerce')
df['Quarter'] = df['Year'].str.extract(r'(Q[1-4])').fillna('Q1').iloc[:, 0].str.replace('Q', '').astype(int)

# Create engineered features that may explain spending behavior.
df['Claims_per_Beneficiary'] = df['Tot_Clms'] / df['Tot_Benes'].replace(0, np.nan)
df['Log_Tot_Clms'] = np.log1p(df['Tot_Clms'])
df['Log_Tot_Benes'] = np.log1p(df['Tot_Benes'])

# Frequency-based proxies for brand and generic popularity.
df['Brand_Frequency'] = df['Brnd_Name'].map(df['Brnd_Name'].value_counts())
df['Generic_Frequency'] = df['Gnrc_Name'].map(df['Gnrc_Name'].value_counts())

# Keep the target in a separate series.
target = 'Tot_Spndng'
feature_cols = [
    'Tot_Benes', 'Tot_Clms', 'Avg_Spnd_Per_Bene', 'Avg_Spnd_Per_Clm',
    'Year_Value', 'Quarter', 'Claims_per_Beneficiary', 'Log_Tot_Clms',
    'Log_Tot_Benes', 'Brand_Frequency', 'Generic_Frequency',
    'Brnd_Name', 'Gnrc_Name'
]

X = df[feature_cols]
y = df[target]

print('Shape after feature engineering:', X.shape)
print('Target summary:')
print(y.describe())


## Train-Test Split and Preprocessing

A pipeline is used so the numeric and categorical features are handled consistently. Numeric columns are imputed and standardized, while categorical columns are one-hot encoded.

In [ ]:
# Split into training and testing data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = [
    'Tot_Benes', 'Tot_Clms', 'Avg_Spnd_Per_Bene', 'Avg_Spnd_Per_Clm',
    'Year_Value', 'Quarter', 'Claims_per_Beneficiary', 'Log_Tot_Clms',
    'Log_Tot_Benes', 'Brand_Frequency', 'Generic_Frequency'
]
categorical_features = ['Brnd_Name', 'Gnrc_Name']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features),
    ]
)

print('Training set size:', len(X_train))
print('Testing set size:', len(X_test))


## Model Building

Two regression models are compared: ordinary least squares regression and ridge regression with regularization. Both are trained using the same preprocessed features for a fair comparison.

In [ ]:
results = []

for model_name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression', Ridge(alpha=2.0))
]:
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')

    results.append({
        'Model': model_name,
        'Test_R2': r2,
        'Test_RMSE': rmse,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std(),
        'Predictions': predictions
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Test_R2', ascending=False)
results_df


## Model Evaluation Summary

The table below compares the regression models using test-set R-squared and RMSE. Cross-validation scores are also included to estimate generalization performance.

In [ ]:
results_df[['Model', 'Test_R2', 'Test_RMSE', 'CV_R2_Mean', 'CV_R2_Std']].round(4)


## Visual Comparison of Performance

The visualizations below help compare how well each model fits the test set and how stable the cross-validation performance is.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_predictions = results_df.loc[results_df['Model'] == best_model_name, 'Predictions'].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot for actual vs predicted values.
for ax, row in zip(axes, results_df.itertuples(index=False)):
    ax.scatter(y_test, row.Predictions, alpha=0.6)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    ax.set_title(f'{row.Model}: Actual vs Predicted Spending')
    ax.set_xlabel('Actual Spending')
    ax.set_ylabel('Predicted Spending')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance bar chart.
metric_df = results_df[['Model', 'Test_R2', 'Test_RMSE']].melt(id_vars='Model', var_name='Metric', value_name='Value')
plt.figure(figsize=(10, 5))
sns.barplot(data=metric_df, x='Model', y='Value', hue='Metric')
plt.title('Regression Model Performance Comparison')
plt.xticks(rotation=30)
plt.ylabel('Score')
plt.tight_layout()
plt.show()


## Interpretation and Insights

The best-performing model is the one with the highest test-set R-squared and the lowest RMSE. The engineered features improve the model by combining behavioral and structural signals in the healthcare claims data.

In [ ]:
best_row = results_df.iloc[0]
print('Best performing model:', best_row['Model'])
print('Test R2:', round(best_row['Test_R2'], 4))
print('Test RMSE:', round(best_row['Test_RMSE'], 2))
print('Cross-validated R2 mean:', round(best_row['CV_R2_Mean'], 4))
print('Cross-validated R2 std:', round(best_row['CV_R2_Std'], 4))

print('Key insight: the engineered features such as claims-per-beneficiary and log-transformed count variables capture strong non-linear and scale effects that help explain spending variation.')
